In [2]:
# =============================================================================
# TANAGER STUDY AREA MAP
# =============================================================================

from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx

from rasterio.transform import from_origin
from rasterio.crs import CRS
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling,
)

from matplotlib_scalebar.scalebar import ScaleBar

# NEW IMPORTS (for GeoJSON export)
import geopandas as gpd
from shapely.geometry import Polygon

# =============================================================================
# FILE PATH
# =============================================================================

DATA_DIR = Path(r"D:\TENAGER HYPERSPECTRAL DATA\JAPAN")

HDF_FILE = DATA_DIR / "20250503_020525_16_4001_ortho_sr_hdf5.h5"

OUTPUT = DATA_DIR / "Study_Area_Map.png"
GEOJSON_OUT = DATA_DIR / "Study_Area_Boundary.geojson"

# =============================================================================
# READ HDF5
# =============================================================================

print("Reading Tanager image...")

with h5py.File(HDF_FILE, "r") as f:
    cube = f["HDFEOS/GRIDS/HYP/Data Fields/surface_reflectance"][:]
    nodata = f["HDFEOS/GRIDS/HYP/Data Fields/nodata_pixels"][:]

cube = np.transpose(cube, (1, 2, 0))
print("Cube shape:", cube.shape)

# =============================================================================
# RGB COMPOSITE
# =============================================================================

rgb = cube[:, :, [60, 35, 15]].astype(np.float32)
rgb[nodata != 0] = np.nan

# =============================================================================
# CONTRAST STRETCH
# =============================================================================

print("Applying percentile stretch...")

for i in range(3):
    band = rgb[:, :, i]
    low = np.nanpercentile(band, 2)
    high = np.nanpercentile(band, 98)
    band = (band - low) / (high - low)
    rgb[:, :, i] = np.clip(band, 0, 1)

gamma = 0.85
rgb = rgb ** gamma

# =============================================================================
# IMAGE INFORMATION
# =============================================================================

pixel_size = 30.0

xmin = 666390.0
xmax = 693240.0
ymax = 3870480.0
ymin = 3850320.0

width = rgb.shape[1]
height = rgb.shape[0]

transform = from_origin(
    xmin,
    ymax,
    pixel_size,
    pixel_size,
)

src_crs = CRS.from_epsg(32653)
dst_crs = CRS.from_epsg(3857)

# =============================================================================
# 🌍 CREATE RASTER BOUNDARY (NEW SECTION)
# =============================================================================

# Corner coordinates in source CRS
boundary_poly = Polygon([
    (xmin, ymin),
    (xmin, ymax),
    (xmax, ymax),
    (xmax, ymin),
    (xmin, ymin)
])

gdf = gpd.GeoDataFrame(
    {"name": ["tanager_study_area"]},
    geometry=[boundary_poly],
    crs=src_crs
)

# Reproject boundary to Web Mercator (match map)
gdf_3857 = gdf.to_crs(dst_crs)

# Save GeoJSON
gdf_3857.to_file(GEOJSON_OUT, driver="GeoJSON")

print("Boundary saved:", GEOJSON_OUT)
print("GeoJSON saved:", GEOJSON_OUT)

Reading Tanager image...
Cube shape: (672, 895, 426)
Applying percentile stretch...
Boundary saved: D:\TENAGER HYPERSPECTRAL DATA\JAPAN\Study_Area_Boundary.geojson
GeoJSON saved: D:\TENAGER HYPERSPECTRAL DATA\JAPAN\Study_Area_Boundary.geojson
